# e2 — Master_v2 construction

Merges:
- `intermediary/master_data_wide.csv` (Master_extended, from e1)
- `intermediary/trade_metrics.csv` (from e1b)
- `rawdata/PopulationWDI.csv`

Applies:
- Sovereign-states filter (drops dependent territories and regional aggregates)
- Current WB high-income classification exclusion (with Gulf-state override)
- Variable cleanup: drops WB rents (replaced by trade shares), 2 Adjusted Savings, 2 Reserves
- Pre-fills: `Civil war = 0`, `Production_* = 0`, `Use of IMF credit = 0`
- `post_2019` dummy and three pre-computed COVID interaction columns
- Two-stage imputation:
  - Stage 1: linear interpolation interior gaps (MAX_GAP=3), ffill trailing (limit=3), bfill leading (limit=3)
  - Stage 2: miceforest with M=10 random-forest imputations, country-demeaned to bias toward within-country variation
- Validation: random-cell masking + per-variable median % error
- Error-weighted reliance score per country

Outputs:
- `intermediary/Master_v2_imputations.parquet` — M=10 stacked panels, keyed by `imputation_id`
- `intermediary/Master_v2_diagnostics.csv` — per-variable validation accuracy + per-country reliance
- `intermediary/Master_v2_observed.csv` — observed cells only (no imputation), for robustness checks

Downstream (e5/e6) uses `_mice_pool.py` for Rubin's-rules pooling.


## 1. Setup

In [ ]:
import os, sys, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='miceforest')

sys.path.insert(0, '.')
import _config as cfg
from standardize_country import ISO3_TO_WB

EXT_DIR  = Path('.').resolve()
INTER    = EXT_DIR / 'intermediary'
RAWDATA  = EXT_DIR / 'rawdata'

print(f'Working dir: {EXT_DIR}')
print(f'YEAR_MIN, YEAR_MAX = {cfg.YEAR_MIN}, {cfg.YEAR_MAX}')


## 2. Sovereign-state filter

In [ ]:
# ISO3 codes that are dependent territories or non-sovereign entities.
# Exclude: HKG, MAC (SARs of China); VAT (microstate with no economy).
# Include: TWN (de facto sovereign), PSE/XKX (UN observers / partial recognition).
DEPENDENT_TERRITORIES = {
    'ANT',  # Netherlands Antilles (dissolved 2010)
    'ABW',  # Aruba
    'AIA',  # Anguilla
    'ASM',  # American Samoa
    'BES',  # Bonaire, Sint Eustatius and Saba
    'BMU',  # Bermuda
    'BVT',  # Bouvet Island
    'CCK',  # Cocos (Keeling) Islands
    'COK',  # Cook Islands
    'CUW',  # Curaçao
    'CXR',  # Christmas Island
    'CYM',  # Cayman Islands
    'ESH',  # Western Sahara
    'FLK',  # Falkland Islands
    'FRO',  # Faroe Islands
    'GGY',  # Guernsey
    'GIB',  # Gibraltar
    'GLP',  # Guadeloupe
    'GRL',  # Greenland
    'GUF',  # French Guiana
    'GUM',  # Guam
    'HKG',  # Hong Kong SAR (excluded per sovereignty principle)
    'IMN',  # Isle of Man
    'IOT',  # British Indian Ocean Territory
    'JEY',  # Jersey
    'MAC',  # Macao SAR
    'MAF',  # Saint Martin (French)
    'MNP',  # Northern Mariana Islands
    'MSR',  # Montserrat
    'MTQ',  # Martinique
    'MYT',  # Mayotte
    'NCL',  # New Caledonia
    'NFK',  # Norfolk Island
    'NIU',  # Niue
    'PCN',  # Pitcairn
    'PRI',  # Puerto Rico
    'PYF',  # French Polynesia
    'REU',  # Réunion
    'SGS',  # South Georgia
    'SHN',  # Saint Helena
    'SJM',  # Svalbard and Jan Mayen
    'SPM',  # Saint Pierre and Miquelon
    'SXM',  # Sint Maarten
    'TCA',  # Turks and Caicos
    'TKL',  # Tokelau
    'UMI',  # US Minor Outlying Islands
    'VAT',  # Vatican City (no economy)
    'VGB',  # British Virgin Islands
    'VIR',  # US Virgin Islands
    'WLF',  # Wallis and Futuna
}

# Gulf state guarantee override (kept despite HIC classification)
GULF_STATES = {'ARE', 'BHR', 'KWT', 'OMN', 'QAT', 'SAU', 'IRQ', 'IRN', 'YEM'}

print(f'Dependent territories blacklist: {len(DEPENDENT_TERRITORIES)} codes')
print(f'Gulf-state override: {len(GULF_STATES)} codes')


In [ ]:
# Hardcoded 1995 World Bank high-income country (HIC) classification.
# Threshold at the time: 1993 GNI per capita > USD 8,955.
# Includes only sovereign states; dependent territories are excluded by the
# preceding territories filter regardless.
# Source: WB Country and Lending Groups historical classification, 1995.
HIC_1995 = {
    'AUS',  # Australia
    'AUT',  # Austria
    'BEL',  # Belgium
    'BHS',  # Bahamas
    'BRN',  # Brunei
    'CAN',  # Canada
    'CHE',  # Switzerland
    'CYP',  # Cyprus
    'DEU',  # Germany
    'DNK',  # Denmark
    'ESP',  # Spain
    'FIN',  # Finland
    'FRA',  # France
    'GBR',  # United Kingdom
    'GRC',  # Greece
    'IRL',  # Ireland
    'ISL',  # Iceland
    'ISR',  # Israel
    'ITA',  # Italy
    'JPN',  # Japan
    'KOR',  # Korea, Rep.
    'LIE',  # Liechtenstein (microstate; will also be dropped by territory list if present)
    'LUX',  # Luxembourg
    'MCO',  # Monaco
    'NLD',  # Netherlands
    'NOR',  # Norway
    'NZL',  # New Zealand
    'PRT',  # Portugal
    'SGP',  # Singapore
    'SMR',  # San Marino
    'SWE',  # Sweden
    'USA',  # United States
    # Gulf states 1995-HIC (kept by Gulf override below)
    'ARE',  # United Arab Emirates
    'BHR',  # Bahrain
    'KWT',  # Kuwait
    'QAT',  # Qatar
    # Other small high-income economies in 1995
    'ATG',  # Antigua and Barbuda
    'BRB',  # Barbados
}

hic_codes = HIC_1995
print(f'1995 HIC list (sovereign): {len(hic_codes)} countries')

# Effective exclusion: HIC minus Gulf override
hic_exclude = hic_codes - GULF_STATES
print(f'HIC after Gulf override: excluding {len(hic_exclude)} countries')


## 3. Load Master_extended and trade_metrics

In [ ]:
# Master_extended (from e1)
M = pd.read_csv(INTER / 'master_data_wide.csv')
print(f'Master_extended: {len(M):,} rows, {len(M.columns)} cols, '
      f'{M["Country Code"].nunique()} countries, {M["Year"].min()}-{M["Year"].max()}')

# Trade metrics (from e1b)
T = pd.read_csv(INTER / 'trade_metrics.csv')
T = T.rename(columns={'reporterISO': 'Country Code', 'period': 'Year'})
T['Year'] = T['Year'].astype(int)
print(f'Trade metrics: {len(T):,} rows, {len(T.columns)} cols, '
      f'{T["Country Code"].nunique()} countries, {T["Year"].min()}-{T["Year"].max()}')

# Clip both to YEAR_MIN..YEAR_MAX
M = M[(M['Year'] >= cfg.YEAR_MIN) & (M['Year'] <= cfg.YEAR_MAX)]
T = T[(T['Year'] >= cfg.YEAR_MIN) & (T['Year'] <= cfg.YEAR_MAX)]
print(f'After year clip: M={len(M):,}, T={len(T):,}')


## 4. Sovereign filter + HIC filter, then merge

In [ ]:
# Apply sovereign + HIC filters to Master_extended
pre_n = M['Country Code'].nunique()
M = M[~M['Country Code'].isin(DEPENDENT_TERRITORIES)]
mid_n = M['Country Code'].nunique()
M = M[~M['Country Code'].isin(hic_exclude)]
post_n = M['Country Code'].nunique()

print(f'Country count: {pre_n} -> {mid_n} after territories -> {post_n} after HIC (Gulf override applied)')
print(f'Master after filters: {len(M):,} rows, {post_n} countries')


# Drop countries with insufficient time-series coverage in the panel.
# These appeared in the audit as having <5 years of trade data:
#   ERI (Eritrea, 1 year), TKM (Turkmenistan, 4 years), TUV (Tuvalu, 4 years)
# Their within-country variation is too sparse for FE identification.
INSUFFICIENT_COVERAGE = {'ERI', 'TKM', 'TUV'}
pre_filter_n = M['Country Code'].nunique()
M = M[~M['Country Code'].isin(INSUFFICIENT_COVERAGE)]
post_filter_n = M['Country Code'].nunique()
print(f'Insufficient-coverage filter: dropped {pre_filter_n - post_filter_n} countries '
      f'({sorted(INSUFFICIENT_COVERAGE)})')


In [ ]:
# ECI filter removed: countries without trade data are dropped at the
# inner merge in the next cell, which is the more principled cut.
print('ECI filter skipped — trade-merge inner join handles country coverage')


In [ ]:
# Merge Master + Trade (inner join on Country Code, Year)
# Inner because we need both sides for the headline analysis
df = pd.merge(M, T, on=['Country Code', 'Year'], how='inner',
              suffixes=('', '_trade'))
print(f'Merged: {len(df):,} rows, {len(df.columns)} cols, {df["Country Code"].nunique()} countries')

# Diagnostic: countries in M but not in T (and vice versa)
m_only = set(M['Country Code'].unique()) - set(T['Country Code'].unique())
t_only = set(T['Country Code'].unique()) - set(M['Country Code'].unique())
print(f'In Master only (no trade data): {len(m_only)}')
print(f'In Trade only (no Master, expected — trade has 207 reporters): {len(t_only)}')


## 5. Population merge

In [ ]:
pop_raw = pd.read_csv(RAWDATA / 'PopulationWDI.csv')
year_cols = [c for c in pop_raw.columns if c.startswith(tuple(str(y) for y in range(1990, 2030)))]
pop_long = pop_raw.melt(
    id_vars=['Series Name', 'Series Code', 'Country Name', 'Country Code'],
    value_vars=year_cols,
    var_name='Year_raw',
    value_name='Population',
)
pop_long['Year'] = pop_long['Year_raw'].str.extract(r'(\d{4})').astype(int)
pop_long['Population'] = pd.to_numeric(pop_long['Population'], errors='coerce')
pop_long = pop_long[['Country Code', 'Year', 'Population']]
pop_long = pop_long[
    (pop_long['Year'] >= cfg.YEAR_MIN) & (pop_long['Year'] <= cfg.YEAR_MAX)
]

df = df.merge(pop_long, on=['Country Code', 'Year'], how='left')
print(f'After population merge: {len(df):,} rows, {len(df.columns)} cols')
print(f'Population NaN: {df["Population"].isna().sum()}')


## 6. Drop deprecated columns

In [ ]:
# Replaced by trade_metrics shares (wide_resource_share, hydrocarbon_share, ores_share)
WB_RENTS_TO_DROP = [
    'Total natural resources rents (% of GDP)',
    'Oil rents (% of GDP)',
    'Natural gas rents (% of GDP)',
    'Mineral rents (% of GDP)',
    'Forestry rents (% of GDP)',
]

# Source-data cliff at 2022; replaced by no current variable
ADJUSTED_SAVINGS_TO_DROP = [
    'Adjusted savings: gross savings (% of GNI)',
    'Adjusted savings: total (current US$)',
    'Adjusted savings: natural resources depletion (% of GNI)',  # also drop the depletion column
]

# >98% missing; not usable as a panel variable
RESERVES_TO_DROP = [
    'Reserves_Metals',
    'Reserves_Others',
]

to_drop = WB_RENTS_TO_DROP + ADJUSTED_SAVINGS_TO_DROP + RESERVES_TO_DROP
existing = [c for c in to_drop if c in df.columns]
missing  = [c for c in to_drop if c not in df.columns]
df = df.drop(columns=existing)
print(f'Dropped {len(existing)} columns: {existing}')
if missing:
    print(f'(not found in df: {missing})')
print(f'After drops: {len(df.columns)} cols')


## 7. Pre-fills: zero where NaN means absence

In [ ]:
# Civil war: V-Dem binary. NaN ~ no recorded conflict in country-year -> 0.
if 'Civil war' in df.columns:
    pre = df['Civil war'].isna().sum()
    df['Civil war'] = df['Civil war'].fillna(0)
    print(f'Civil war: filled {pre} NaN with 0')

# Production_* columns: countries with no production in this category genuinely
# produce zero, not "unknown". Distinguishing requires a coverage flag.
for col in ['Production_Hydrocarbons', 'Production_Metals', 'Production_Others']:
    if col in df.columns:
        pre = df[col].isna().sum()
        df[col] = df[col].fillna(0)
        print(f'{col}: filled {pre} NaN with 0')

# Use of IMF credit: NaN ~ not in IMF database = no outstanding IMF debt = 0.
# (Faithful to v2 logic.)
if 'Use of IMF credit (DOD, current US$)' in df.columns:
    pre = df['Use of IMF credit (DOD, current US$)'].isna().sum()
    df['Use of IMF credit (DOD, current US$)'] = df['Use of IMF credit (DOD, current US$)'].fillna(0)
    print(f'Use of IMF credit: filled {pre} NaN with 0')

print()
print('NB: Death rates pre-fill from v2 is NOT applied — NaN means missing, not zero deaths.')


## 8. `post_2019` dummy + pre-computed COVID interactions

In [ ]:
df['post_2019'] = (df['Year'] >= 2020).astype(int)

# Pre-compute interactions: post_2019 × {hydrocarbon, ores, base_metals}_share
for share_col, name in [
    ('hydrocarbon_share', 'post2019_x_hydrocarbon_share'),
    ('ores_share',         'post2019_x_ores_share'),
    ('base_metals_share',  'post2019_x_base_metals_share'),
    # Disaggregated HS27 sub-codes
    ('coal_share',         'post2019_x_coal_share'),
    ('crude_oil_share',    'post2019_x_crude_oil_share'),
    ('refined_oil_share',  'post2019_x_refined_oil_share'),
    ('gas_share',          'post2019_x_gas_share'),
]:
    if share_col in df.columns:
        df[name] = df['post_2019'] * df[share_col]
        print(f'  Added {name}')
    else:
        print(f'  WARNING: {share_col} not in df, skipping {name}')


## 9. Stage 1 imputation: per-country interpolation

- Interior gaps (within a country's time series): linear interpolation, MAX_GAP=3
- Leading gaps (first observed value > 3 years into panel): bfill, limit=3
- Trailing gaps (last observed value > 3 years before panel end): ffill, limit=3

Avoids COVID-era linear extrapolation artifacts. Variables with no observations
at all in a country are left NaN for Stage 2 (MICE) to handle cross-country.

In [ ]:
MAX_GAP = 3

# Variables to EXCLUDE from interpolation (time-invariant, dummies, pre-filled)
EXCLUDE_FROM_INTERP = {
    'Country Code', 'Country Name', 'Year', 'Population',
    'Landlocked', 'post_2019', 'Civil war',
    'Hydrocarbons_Dominant', 'Subsoil_Metals_Dominant', 'Precious_Metals_Dominant',
    'Production_Hydrocarbons', 'Production_Metals', 'Production_Others',
    'Use of IMF credit (DOD, current US$)',
}

# Trade USD aggregates: large, scale-dependent. Better imputed cross-country
# only if absolutely necessary; for now, exclude from Stage 1 (can leave NaN).
TRADE_USD_COLS = [c for c in df.columns if c.endswith('_usd')] + [
    c for c in df.columns if c.isdigit()  # HS chapter raw value cols if any survived
]
EXCLUDE_FROM_INTERP.update(TRADE_USD_COLS)

interp_cols = [c for c in df.columns if c not in EXCLUDE_FROM_INTERP]
print(f'Interpolating {len(interp_cols)} columns')

# Track pre-Stage-1 NaN count
pre_nan = df[interp_cols].isna().sum().sum()

df = df.sort_values(['Country Code', 'Year']).reset_index(drop=True)

for col in interp_cols:
    # Per-country interpolation with three modes
    def _stage1(group):
        s = group[col].copy()
        # Interior: linear, limited gap
        s = s.interpolate(method='linear', limit=MAX_GAP, limit_area='inside')
        # Trailing: ffill, limit MAX_GAP
        s = s.ffill(limit=MAX_GAP)
        # Leading: bfill, limit MAX_GAP
        s = s.bfill(limit=MAX_GAP)
        return s

    df[col] = df.groupby('Country Code', group_keys=False)[col].transform(
        lambda s: (
            s.interpolate(method='linear', limit=MAX_GAP, limit_area='inside')
             .ffill(limit=MAX_GAP)
             .bfill(limit=MAX_GAP)
        )
    )

post_nan = df[interp_cols].isna().sum().sum()
print(f'Stage 1: NaN reduced {pre_nan:,} -> {post_nan:,} ({pre_nan - post_nan:,} filled)')
print(f'Remaining NaN cells: {post_nan:,} (for Stage 2 MICE to handle)')


## 10. Stage 2 imputation: miceforest M=10 with country-demeaning

- Library: `miceforest` (random forest conditional model, native multiple imputation)
- M = 10 imputations with deterministic seeds 0..9
- Pre-step: subtract country mean from each numeric column → run MICE on
  deviations-from-country-mean → add country mean back. This bias toward
  within-country variation addresses the pooled-panel-KNN critique.
- Categorical columns (`Landlocked`, dominance dummies, `post_2019`, `Civil war`)
  are passed as `category` dtype so miceforest uses appropriate splits.

In [ ]:
# Check for miceforest
try:
    import miceforest as mf
    print(f'miceforest version: {mf.__version__}')
except ImportError:
    print('miceforest not installed. Install with:')
    print('  pip install miceforest --break-system-packages')
    raise

M_IMPUTATIONS = 5  # was 10; reduced for speed, re-run with 10 for paper


In [ ]:
# Columns to pass to MICE
ID_COLS = ['Country Code', 'Country Name', 'Year']
SKIP_FROM_MICE = {
    # Don't synthesize the forecast benchmark
    'Atlas growth projection',
    # Already filled in pre-fill stage
    'Civil war', 'Production_Hydrocarbons', 'Production_Metals', 'Production_Others',
    'Use of IMF credit (DOD, current US$)',
    # Trade USD aggregates: scale-dependent, leave NaN; share columns will be imputed
}
SKIP_FROM_MICE.update(TRADE_USD_COLS)

CATEGORICAL_COLS = {
    'Landlocked', 'post_2019',
    'Hydrocarbons_Dominant', 'Subsoil_Metals_Dominant', 'Precious_Metals_Dominant',
}
CATEGORICAL_COLS &= set(df.columns)

mice_cols = [c for c in df.columns
             if c not in ID_COLS
             and c not in SKIP_FROM_MICE
             and c not in CATEGORICAL_COLS]
print(f'MICE numeric columns: {len(mice_cols)}')
print(f'MICE categorical columns: {len(CATEGORICAL_COLS)}')

# Pre-step: country-demeaning on numeric columns
country_means = df.groupby('Country Code')[mice_cols].transform('mean')
df_demeaned = df.copy()
df_demeaned[mice_cols] = df[mice_cols] - country_means

# Cast categorical columns
for c in CATEGORICAL_COLS:
    df_demeaned[c] = df_demeaned[c].astype('category')

# LightGBM (under miceforest) does not accept JSON special characters in
# column names: ":", ",", "(", ")", "[", "]", "{", "}", quotes, slashes.
# Sanitise on the way in, restore on the way out.
import re as _re
def _sanitise(name):
    s = _re.sub(r'[^A-Za-z0-9_]', '_', name)
    return _re.sub(r'_+', '_', s).strip('_')

SAN_MAP = {c: _sanitise(c) for c in mice_cols + list(CATEGORICAL_COLS)}
# Check for sanitisation collisions
collisions = [k for k, v in SAN_MAP.items() if list(SAN_MAP.values()).count(v) > 1]
if collisions:
    print(f'WARNING: {len(collisions)} name collisions after sanitisation. Adding hash suffix.')
    # Add hash to disambiguate
    seen = {}
    for k in list(SAN_MAP.keys()):
        v = SAN_MAP[k]
        if v in seen.values():
            SAN_MAP[k] = f'{v}_{abs(hash(k)) % 10000:04d}'
        seen[k] = SAN_MAP[k]
SAN_INV = {v: k for k, v in SAN_MAP.items()}

print(f'Demeaning: country means computed. Shape: {df_demeaned.shape}')
print(f'Sanitised {len(SAN_MAP)} column names for MICE')


In [ ]:
# Pre-MICE diagnostic: where are NaNs now?
pre_mice_nan = df_demeaned[mice_cols + list(CATEGORICAL_COLS)].isna().sum()
pre_mice_nan = pre_mice_nan[pre_mice_nan > 0].sort_values(ascending=False)
print('Remaining NaN cells per column (top 20, going into MICE):')
print(pre_mice_nan.head(20).to_string())
print(f'\nTotal NaN: {pre_mice_nan.sum():,}')
print(f'Total cells: {len(df_demeaned) * (len(mice_cols) + len(CATEGORICAL_COLS)):,}')
print(f'Missing rate: {100 * pre_mice_nan.sum() / (len(df_demeaned) * (len(mice_cols) + len(CATEGORICAL_COLS))):.2f}%')


In [ ]:
# Run MICE
# M_IMPUTATIONS was 10; reduced to 5 to halve runtime. Rubin's rules still
# apply with M=5; SE noise is modest when FMI < 0.3. Re-run with M=10 for
# paper-quality numbers once the pipeline is validated.
M_IMPUTATIONS = 5
ITERATIONS = 5

mice_input = df_demeaned[mice_cols + list(CATEGORICAL_COLS)].copy()
# Apply name sanitisation
mice_input = mice_input.rename(columns=SAN_MAP)

# ── OPTIMIZATION: only impute columns that contain NaN, and restrict
# predictors per target to the top-K most correlated other columns.
# Columns with no missing values are still passed to miceforest (it needs
# the full data matrix for prediction) but they're not listed as targets,
# so no RF is trained for them. Substantial speed-up.

nan_counts = mice_input.isna().sum()
cols_with_missing = nan_counts[nan_counts > 0].index.tolist()
cols_no_missing = nan_counts[nan_counts == 0].index.tolist()

print(f'Columns with missing values: {len(cols_with_missing)}')
print(f'Columns fully observed (skipped as targets): {len(cols_no_missing)}')

TOP_K_PREDICTORS = 20

# Compute correlation matrix on numeric columns only (categoricals will be added back)
cat_sanitised = {SAN_MAP[c] for c in CATEGORICAL_COLS if c in SAN_MAP}
numeric_for_corr = [c for c in mice_input.columns if c not in cat_sanitised]

corr = mice_input[numeric_for_corr].corr().abs()

# Build variable_schema: for each target, its top-K predictors (excluding itself).
# All categorical columns are always included as candidate predictors.
variable_schema = {}
for target in cols_with_missing:
    if target in cat_sanitised:
        predictors = [c for c in mice_input.columns if c != target]
    else:
        if target not in corr.columns:
            continue
        top = corr[target].drop(target).sort_values(ascending=False).head(TOP_K_PREDICTORS).index.tolist()
        predictors = top + list(cat_sanitised)
    variable_schema[target] = [p for p in predictors if p in mice_input.columns]

print(f'Variable schema: {len(variable_schema)} targets, '
      f'~{int(np.mean([len(v) for v in variable_schema.values()]))} predictors each (median)')

print(f'\nStarting miceforest: M={M_IMPUTATIONS} imputations, '
      f'{len(mice_input.columns)} total cols (imputing {len(variable_schema)}), '
      f'{len(mice_input):,} rows, {ITERATIONS} iterations')

kernel = mf.ImputationKernel(
    mice_input,
    num_datasets=M_IMPUTATIONS,
    variable_schema=variable_schema,
    random_state=0,
)

# Per-iteration progress: call .mice(iterations=1) in a loop so we get
# timing and ETA printed between each pass. miceforest preserves state
# across multiple .mice() calls so this is equivalent to a single call
# with iterations=N but with visibility.
t0 = time.time()
for it in range(1, ITERATIONS + 1):
    t_iter = time.time()
    kernel.mice(iterations=1, verbose=False)
    iter_time = time.time() - t_iter
    elapsed = time.time() - t0
    remaining_eta = (ITERATIONS - it) * iter_time
    print(f'  iter {it}/{ITERATIONS}  ({iter_time:5.1f}s, '
          f'elapsed {elapsed/60:4.1f} min, ETA {remaining_eta/60:4.1f} min)')

print(f'\nMICE done in {(time.time() - t0)/60:.1f} min')

# Extract all M imputed datasets, undo sanitisation
imputed_datasets = []
for m in range(M_IMPUTATIONS):
    sub = kernel.complete_data(dataset=m).copy()
    # Restore original column names
    sub = sub.rename(columns=SAN_INV)
    # Reattach ID cols
    sub = pd.concat([
        df[ID_COLS].reset_index(drop=True),
        sub.reset_index(drop=True)
    ], axis=1)
    # Add back country means (undo demeaning) for numeric columns
    for c in mice_cols:
        sub[c] = sub[c] + country_means[c].reset_index(drop=True)
    sub['imputation_id'] = m
    imputed_datasets.append(sub)

print(f'Extracted {len(imputed_datasets)} imputed panels.')


# Post-MICE: clip imputed values to plausible ranges. MICE on demeaned data
# occasionally produces values outside the observed range (e.g., negative
# GFCF as % of GDP); clip them back to physically meaningful bounds.
# Observed-only cells are untouched since they're not imputed.
POST_MICE_CLIPS = {
    'Gross fixed capital formation, all, Constant prices, Percent of GDP': (0, None),
    'Trade (% of GDP)': (0, None),
    'Government revenue': (0, None),
    'Domestic credit to private sector (% of GDP)': (0, None),
    'Urban population (% of total population)': (0, 100),
    'Access to electricity (% of population)': (0, 100),
    'Life expectancy at birth, total (years)': (0, 100),
    'Mobile cellular subscriptions (per 100 people)': (0, None),
    'Inflation, consumer prices (annual %)': (-30, None),  # observed min was -16.86
    'Human capital index': (1.0, 4.0),
    # Share variables: defensively clip to [0, 1]
    'wide_resource_share': (0, 1),
    'hydrocarbon_share': (0, 1),
    'ores_share': (0, 1),
    'base_metals_share': (0, 1),
    'precious_share': (0, 1),
    'tight_share': (0, 1),
    'extractives_share': (0, 1),
    # The COVID interactions are products of share × dummy so also in [0, 1]
    'post2019_x_hydrocarbon_share': (0, 1),
    'post2019_x_ores_share': (0, 1),
    'post2019_x_base_metals_share': (0, 1),
    # Disaggregated HS27 sub-code shares
    'coal_share':                    (0, 1),
    'crude_oil_share':               (0, 1),
    'refined_oil_share':             (0, 1),
    'gas_share':                     (0, 1),
    'post2019_x_coal_share':         (0, 1),
    'post2019_x_crude_oil_share':    (0, 1),
    'post2019_x_refined_oil_share':  (0, 1),
    'post2019_x_gas_share':          (0, 1),
}

clip_summary = []
for ds in imputed_datasets:
    for col, (lo, hi) in POST_MICE_CLIPS.items():
        if col not in ds.columns:
            continue
        before = ds[col].copy()
        if lo is not None:
            ds[col] = ds[col].clip(lower=lo)
        if hi is not None:
            ds[col] = ds[col].clip(upper=hi)
        n_clipped = (before != ds[col]).sum()
        if n_clipped > 0:
            clip_summary.append((col, n_clipped))

# Per-column summary across all M imputations
from collections import Counter
clip_totals = Counter()
for col, n in clip_summary:
    clip_totals[col] += n

print(f'Post-MICE clipping applied:')
for col, n in clip_totals.most_common():
    print(f'  {col}: {n} cells clipped (across all M)')
if not clip_totals:
    print('  No cells required clipping.')


## 11. Validation: random-mask + per-variable median % error

In [ ]:
# For each numeric column, mask 5% of known values, predict, compute median % error.
# Uses a single MICE run for speed; this is the MAR-style validation. MNAR is the
# next iteration's work.

np.random.seed(0)
MASK_FRAC = 0.05
# mice_input was renamed to sanitised column names in Section 10 before
# being passed to LightGBM. Restore originals here so the loop below can
# index by mice_cols (which holds original names). mask_input is re-sanitised
# just before val_kernel further down this cell.
mask_input = mice_input.copy().rename(columns=SAN_INV)
masked = {}  # col -> [(row_idx, true_value), ...]

for col in mice_cols:
    obs_idx = mask_input[mask_input[col].notna()].index.tolist()
    n_mask = int(len(obs_idx) * MASK_FRAC)
    if n_mask == 0:
        continue
    sample = np.random.choice(obs_idx, size=n_mask, replace=False)
    truth = mask_input.loc[sample, col].copy()
    mask_input.loc[sample, col] = np.nan
    masked[col] = (sample, truth)

print(f'Masked {sum(len(v[0]) for v in masked.values()):,} cells across {len(masked)} columns')

# Run single-imputation validation
mask_input_san = mask_input.rename(columns=SAN_MAP)
val_kernel = mf.ImputationKernel(mask_input_san, num_datasets=1, random_state=42)
val_kernel.mice(iterations=5, verbose=False)
val_filled = val_kernel.complete_data(dataset=0).rename(columns=SAN_INV)

# Compute median % error per column
validation_rows = []
for col, (idx, truth) in masked.items():
    pred = val_filled.loc[idx, col]
    # Median absolute percent error
    mask_nonzero = truth.abs() > 1e-8
    if mask_nonzero.sum() == 0:
        median_pct_error = np.nan
    else:
        pct_err = np.abs(pred[mask_nonzero] - truth[mask_nonzero]) / np.abs(truth[mask_nonzero]) * 100
        median_pct_error = float(pct_err.median())
    mae = float((pred - truth).abs().median())
    validation_rows.append({
        'variable': col,
        'n_masked': len(idx),
        'median_pct_error': median_pct_error,
        'median_abs_error': mae,
    })

validation_df = pd.DataFrame(validation_rows).sort_values('median_pct_error', ascending=False)
print('\nValidation (top 15 by error):')
print(validation_df.head(15).to_string(index=False))
print('\nValidation (bottom 10 by error, most reliable):')
print(validation_df.tail(10).to_string(index=False))


## 12. Error-weighted reliance score

In [ ]:
# Per country, what fraction of cells were filled by Stage 2, and weight by
# per-variable validation error. High weighted reliance = country's data is
# more synthetic AND that synthesis is unreliable.

# Recompute: which cells were NaN before MICE?
pre_mice_mask = df_demeaned[mice_cols + list(CATEGORICAL_COLS)].isna()

# Per-country, per-variable count of imputed cells
fill_counts = pd.concat([
    df[['Country Code']].reset_index(drop=True),
    pre_mice_mask.reset_index(drop=True),
], axis=1).groupby('Country Code').sum()

# Per-variable median % error map (numeric vars only; categoricals have NaN)
err_map = dict(zip(validation_df['variable'], validation_df['median_pct_error']))

# Weighted reliance: sum over variables of (filled_cells × pct_error_for_that_var)
# Then divide by total cells in that country.
total_cells_per_country = df.groupby('Country Code').size() * (len(mice_cols) + len(CATEGORICAL_COLS))

reliance_records = []
for cc in fill_counts.index:
    row = fill_counts.loc[cc]
    weighted_score = 0.0
    raw_count = 0
    for col, n in row.items():
        if n == 0:
            continue
        raw_count += n
        err = err_map.get(col, np.nan)
        if pd.notna(err):
            weighted_score += n * err
    raw_pct = 100 * raw_count / total_cells_per_country[cc]
    weighted_pct = weighted_score / total_cells_per_country[cc] if total_cells_per_country[cc] else 0
    reliance_records.append({
        'Country Code': cc,
        'raw_imputed_pct':    raw_pct,
        'weighted_score':      weighted_pct,
    })

reliance_df = pd.DataFrame(reliance_records).sort_values('weighted_score', ascending=False)
print('Top 15 countries by weighted reliance (most-synthetic data):')
print(reliance_df.head(15).to_string(index=False))
print()
print('Bottom 10 countries by weighted reliance (most-observed data):')
print(reliance_df.tail(10).to_string(index=False))


## 13. Write outputs

In [ ]:
# Stack the M datasets into one parquet
stacked = pd.concat(imputed_datasets, ignore_index=True)
print(f'Stacked: {len(stacked):,} rows ({M_IMPUTATIONS} x {len(stacked) // M_IMPUTATIONS} country-years), '
      f'{len(stacked.columns)} cols')

out_parquet = INTER / 'Master_v2_imputations.parquet'
stacked.to_parquet(out_parquet, index=False)
print(f'Wrote {out_parquet} ({out_parquet.stat().st_size / 1e6:.1f} MB)')

# Diagnostic CSV: per-variable accuracy + per-country reliance
diag = {
    'validation': validation_df,
    'reliance': reliance_df,
}
out_csv = INTER / 'Master_v2_diagnostics.csv'
with open(out_csv, 'w') as f:
    f.write('# Per-variable validation (MAR-style random masking, 5%)\n')
    validation_df.to_csv(f, index=False)
    f.write('\n# Per-country reliance (raw + error-weighted)\n')
    reliance_df.to_csv(f, index=False)
print(f'Wrote {out_csv}')

# Observed-only panel (no imputation) for downstream robustness
observed_only = df.copy()
out_observed = INTER / 'Master_v2_observed.csv'
observed_only.to_csv(out_observed, index=False)
print(f'Wrote {out_observed} ({len(observed_only):,} rows, {len(observed_only.columns)} cols, '
      f'NaN cells: {observed_only.isna().sum().sum():,})')


## 14. Summary

In [ ]:
print('=' * 70)
print('Master_v2 construction summary')
print('=' * 70)
print(f'Year range:           {cfg.YEAR_MIN}-{cfg.YEAR_MAX}')
print(f'Countries:            {df["Country Code"].nunique()}')
print(f'Country-years:        {len(df):,}')
print(f'Columns:              {len(df.columns)}')
print(f'Imputations (M):      {M_IMPUTATIONS}')
print(f'Stacked rows:         {len(stacked):,}')
print()
print(f'Files:')
print(f'  intermediary/Master_v2_imputations.parquet  (M=10 stacked)')
print(f'  intermediary/Master_v2_diagnostics.csv      (validation + reliance)')
print(f'  intermediary/Master_v2_observed.csv         (observed-only, for robustness)')
print()
print('Next: e3_clusters / e4_ml / e5_regressions / e6_forecast — all need to')
print('loop over imputation_id and pool results via _mice_pool.py utility.')
